## Notebook to infer cell volumes from morphological features reported for the Saccharomycotina subphylum in Chavez et al., 2023

#### By Marco Fumasoni

In [1]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import os

### Functions

In [2]:
L_COL = "Length avg (μm)"
W_COL = "Width avg (μm)"

SHAPE_COLS = [
    "Spherical (or globose)", "Ovoid", "Elongate", "Ellipsoidal", "Cylindrical",
    "Apiculate", "Bacilliform", "Allantoid", "Lunate", "Triangular", "Irregular",
    "Polymorphic", "Clavate (or club shaped)", "Aculeate (or acicular)", "Tapered",
    "Fusiform", "Bowling pin", "Teardrop", "Curved", "Ogival", "Rectangular"
]

# Unit note: 1 μm³ == 1 fL, so we keep a factor for clarity
UM3_TO_FL = 1.0


def _safe(v):
    try:
        return v if np.isfinite(v) and v >= 0 else np.nan
    except:
        return np.nan

def vol_ellipsoid(L, W):
    a, b = L/2.0, W/2.0
    return _safe((4.0/3.0) * math.pi * a * b * b)

def vol_sphere_from_LW(L, W):
    d = (L + W) / 2.0
    r = d / 2.0
    return _safe((4.0/3.0) * math.pi * r**3)

def vol_cylinder(L, W):
    r = W/2.0
    return _safe(math.pi * r**2 * max(L, 0.0))

def vol_spherocylinder(L, W):
    r = W/2.0
    cyl_h = max(L - 2.0*r, 0.0)
    return _safe(math.pi * r**2 * cyl_h + (4.0/3.0)*math.pi*r**3)

def vol_apiculate(L, W):
    r = W/2.0
    cyl_h = max(L - (r + r), 0.0)  # hemisphere height r + cone height r
    hemi = (2.0/3.0) * math.pi * r**3
    cone = (1.0/3.0) * math.pi * r**2 * r
    return _safe(math.pi * r**2 * cyl_h + hemi + cone)

def vol_curved_rod(L, W, factor=0.95):
    return _safe(factor * vol_spherocylinder(L, W))

def vol_allantoid(L, W):
    return vol_curved_rod(L, W, factor=0.95)

def vol_lunate(L, W):
    return _safe(0.75 * vol_ellipsoid(L, W))

def vol_triangular_prism(L, W):
    return _safe((math.sqrt(3)/4.0) * W**2 * max(L, 0.0))

def vol_rectangular_prism(L, W):
    return _safe(L * W * W)

def vol_frustum(L, W, r2_ratio=0.6):
    r1 = W/2.0
    r2 = max(r2_ratio * r1, 0.0)
    h = max(L, 0.0)
    return _safe((math.pi * h / 3.0) * (r1*r1 + r1*r2 + r2*r2))

def vol_clavate(L, W):
    r = W/2.0
    frustum_h = max(L - r, 0.0)
    hemi = (2.0/3.0) * math.pi * r**3
    fr = vol_frustum(frustum_h, W, r2_ratio=0.6)
    return _safe(hemi + fr)

def vol_acicular(L, W):
    r = W/2.0
    h = max(L, 0.0)
    return _safe((1.0/3.0) * math.pi * r**2 * h)

def vol_tapered(L, W):
    return vol_frustum(L, W, r2_ratio=0.5)

def vol_fusiform(L, W):
    return vol_ellipsoid(L, W)

def vol_bowling_pin(L, W):
    return vol_clavate(L, W)

def vol_teardrop(L, W):
    r = W/2.0
    cone_h = max(L - W, 0.0)
    sphere = (4.0/3.0) * math.pi * r**3
    cone  = (1.0/3.0) * math.pi * r**2 * cone_h
    return _safe(sphere + cone)

def vol_ogival(L, W):
    return vol_apiculate(L, W)

def vol_irregular(L, W):
    return vol_ellipsoid(L, W)

def vol_polymorphic(L, W):
    return vol_ellipsoid(L, W)

VOLUME_FUNCS = {
    "Spherical (or globose)": vol_sphere_from_LW,
    "Ovoid": vol_ellipsoid,
    "Elongate": vol_spherocylinder,
    "Ellipsoidal": vol_ellipsoid,
    "Cylindrical": vol_cylinder,
    "Apiculate": vol_apiculate,
    "Bacilliform": vol_spherocylinder,
    "Allantoid": vol_allantoid,
    "Lunate": vol_lunate,
    "Triangular": vol_triangular_prism,
    "Irregular": vol_irregular,
    "Polymorphic": vol_polymorphic,
    "Clavate (or club shaped)": vol_clavate,
    "Aculeate (or acicular)": vol_acicular,
    "Tapered": vol_tapered,
    "Fusiform": vol_fusiform,
    "Bowling pin": vol_bowling_pin,
    "Teardrop": vol_teardrop,
    "Curved": vol_curved_rod,
    "Ogival": vol_ogival,
    "Rectangular": vol_rectangular_prism,
}

def add_shape_volumes_fL(V: pd.DataFrame,
                         length_col: str = L_COL,
                         width_col: str = W_COL,
                         shape_cols = SHAPE_COLS):
    """
    Computes volumes (in fL) for each row/shape==1 and adds:
      - volume_list_fL    : list of per-shape volumes in fL
      - volume_shapes     : matching shape names
      - volume_1_fL..k_fL : one column per detected shape (order = shape_cols)
      - volume_average_fL : mean of per-row volumes (ignoring NaNs)
    """
    V = V.copy()

    volume_lists = []
    shape_lists  = []

    Ls = V[length_col].astype(float)
    Ws = V[width_col].astype(float)

    for idx, row in V.iterrows():
        L = Ls.at[idx]
        W = Ws.at[idx]
        vols = []
        names = []

        if not np.isfinite(L) or not np.isfinite(W) or L < 0 or W < 0:
            volume_lists.append([np.nan])
            shape_lists.append([])
            continue

        for col in shape_cols:
            if col in row and row[col] == 1:
                fn = VOLUME_FUNCS.get(col, vol_ellipsoid)
                try:
                    vol_um3 = fn(L, W)
                    vol_fl  = vol_um3 * UM3_TO_FL  # identity, documented
                except Exception:
                    vol_fl = np.nan
                vols.append(vol_fl)
                names.append(col)

        if not vols:
            vols = [np.nan]

        volume_lists.append(vols)
        shape_lists.append(names)

    V["volume_list_fL"] = volume_lists
    V["volume_shapes"]  = shape_lists

    max_k = max(len(vs) for vs in volume_lists)
    for k in range(max_k):
        V[f"volume_{k+1}_fL"] = [vs[k] if len(vs) > k else np.nan for vs in volume_lists]

    # Average across the computed volumes in fL (ignore NaNs)
    V["volume_average_fL"] = [
        np.nanmean(vs) if np.any(np.isfinite(vs)) else np.nan
        for vs in volume_lists
    ]

    return V
